# 🧠 สถาปัตยกรรมไปป์ไลน์การตรวจจับวัตถุ (Object Detection Pipeline Architecture)

ยินดีต้อนรับสู่สมุดบันทึกคำอธิบายเชิงปฏิบัติสำหรับ **สถาปัตยกรรมไปป์ไลน์การตรวจจับวัตถุ**! ในสมุดบันทึกนี้ เราจะ:
1. ทำความเข้าใจบทบาทขององค์ประกอบโครงสร้างหลักทั้งสามส่วน: **Backbone**, **Neck** และ **Head**
2. สร้างแบบจำลองไปป์ไลน์การตรวจจับวัตถุที่สมบูรณ์จากศูนย์ (from scratch) ด้วย NumPy
3. จำลองคุณลักษณะหลายระดับขนาด (multiscale features - P3, P4, P5) และการรวมคุณลักษณะแบบ lateral/top-down ของ FPN neck
4. ออกแบบ decoupled heads เพื่อคาดการณ์คลาส (class) และพิกัดถดถอย (regression coordinates)
5. กรองกล่องตัวเลือกโดยใช้ Non-Maximum Suppression (NMS) แยกตามคลาส
6. พลอตแผนที่ลักษณะเด่น (feature maps) และผลลัพธ์เพื่อทำความเข้าใจกระบวนการทำงานของตัวตรวจจับแบบขั้นตอนเดียว (single-stage detectors เช่น YOLO) ด้วยภาพ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
np.random.seed(42)

## 1. การสร้างคลาสไปป์ไลน์ (Pipeline Class Implementation)

เราสร้างคลาส `ObjectDetectionPipeline` ขึ้นเพื่อจำลองการทำงานในแต่ละขั้นตอนของการส่งผ่านข้อมูลไปข้างหน้า (forward pass)

In [ ]:
class ObjectDetectionPipeline:
    def __init__(self, image_size=(640, 640), num_classes=3, iou_threshold=0.5, score_threshold=0.85):
        self.image_size = image_size
        self.num_classes = num_classes
        self.iou_threshold = iou_threshold
        self.score_threshold = score_threshold

    def backbone(self, x):
        # Simulates P3 (stride 8), P4 (stride 16), P5 (stride 32)
        p3_h = self.image_size[0] // 8
        p3_w = self.image_size[1] // 8
        p4_h = self.image_size[0] // 16
        p4_w = self.image_size[1] // 16
        p5_h = self.image_size[0] // 32
        p5_w = self.image_size[1] // 32
        
        p3 = np.random.randn(p3_h, p3_w, 128)
        p4 = np.random.randn(p4_h, p4_w, 256)
        p5 = np.random.randn(p5_h, p5_w, 512)
        return {"P3": p3, "P4": p4, "P5": p5}

    def neck(self, features):
        p3 = features["P3"]
        p4 = features["P4"]
        p5 = features["P5"]
        
        # FPN top-down connection: upsample P5 and add to P4
        p5_upsampled = np.repeat(np.repeat(p5, 2, axis=0), 2, axis=1)
        f4 = p4 + p5_upsampled[:, :, :256]
        
        # Upsample f4 and add to P3
        f4_upsampled = np.repeat(np.repeat(f4, 2, axis=0), 2, axis=1)
        f3 = p3 + f4_upsampled[:, :, :128]
        
        return {"F3": f3, "F4": f4, "F5": p5}

    def head(self, fused_features):
        candidate_boxes = []
        candidate_scores = []
        candidate_class_ids = []
        
        for scale_name, feat in fused_features.items():
            H, W, C = feat.shape
            stride = self.image_size[0] // H
            
            cls_logits = np.random.uniform(-5.0, 5.0, size=(H, W, self.num_classes))
            scores = 1.0 / (1.0 + np.exp(-cls_logits))
            
            bbox_preds = np.zeros((H, W, 4))
            for y in range(H):
                for x in range(W):
                    cx = x * stride + stride / 2
                    cy = y * stride + stride / 2
                    w = np.random.uniform(50.0, 200.0)
                    h = np.random.uniform(50.0, 200.0)
                    bbox_preds[y, x] = [cx - w/2, cy - h/2, cx + w/2, cy + h/2]
            
            for y in range(H):
                for x in range(W):
                    box_scores = scores[y, x]
                    class_id = np.argmax(box_scores)
                    max_score = box_scores[class_id]
                    
                    if max_score >= self.score_threshold:
                        candidate_boxes.append(bbox_preds[y, x])
                        candidate_scores.append(max_score)
                        candidate_class_ids.append(class_id)
                        
        return np.array(candidate_boxes), np.array(candidate_scores), np.array(candidate_class_ids)

    def nms_postprocess(self, boxes, scores, class_ids):
        if len(boxes) == 0:
            return [], [], []
            
        keep_final = []
        for c in range(self.num_classes):
            class_mask = (class_ids == c)
            if not np.any(class_mask):
                continue
                
            c_boxes = boxes[class_mask]
            c_scores = scores[class_mask]
            c_indices = np.where(class_mask)[0]
            
            x1 = c_boxes[:, 0]
            y1 = c_boxes[:, 1]
            x2 = c_boxes[:, 2]
            y2 = c_boxes[:, 3]
            areas = (x2 - x1) * (y2 - y1)
            
            order = c_scores.argsort()[::-1]
            c_keep = []
            
            while order.size > 0:
                i = order[0]
                c_keep.append(c_indices[i])
                
                if order.size == 1:
                    break
                    
                xx1 = np.maximum(x1[i], x1[order[1:]])
                yy1 = np.maximum(y1[i], y1[order[1:]])
                xx2 = np.minimum(x2[i], x2[order[1:]])
                yy2 = np.minimum(y2[i], y2[order[1:]])
                
                w = np.maximum(0.0, xx2 - xx1)
                h = np.maximum(0.0, yy2 - yy1)
                intersection = w * h
                
                union = areas[i] + areas[order[1:]] - intersection
                iou = intersection / union
                
                inds = np.where(iou <= self.iou_threshold)[0]
                order = order[inds + 1]
                
            keep_final.extend(c_keep)
            
        return boxes[keep_final], scores[keep_final], class_ids[keep_final]

    def predict(self, raw_image):
        features = self.backbone(raw_image)
        fused = self.neck(features)
        boxes, scores, class_ids = self.head(fused)
        final_boxes, final_scores, final_class_ids = self.nms_postprocess(boxes, scores, class_ids)
        return final_boxes, final_scores, final_class_ids

## 2. การรันอินเฟอเรนซ์จำลอง (Running Dummy Inference)

มาทำการส่งผ่านข้อมูลไปข้างหน้า (forward pass) ผ่านไปป์ไลน์จำลองของเรากัน

In [ ]:
pipeline = ObjectDetectionPipeline(num_classes=3, score_threshold=0.95, iou_threshold=0.45)
dummy_image = np.random.randint(0, 255, size=(640, 640, 3), dtype=np.uint8)

final_boxes, final_scores, final_class_ids = pipeline.predict(dummy_image)
print(f"Detected {len(final_boxes)} high-confidence components after NMS.")

## 3. การแสดงภาพการไหลของเครือข่ายและปิรามิดระดับขนาด (Visualizing Network Flow and Scale Pyramids)

เราแสดงภาพกริดหลายระดับขนาด (multi-scale grids) ซึ่งเป็นตัวแทนของแผนที่ลักษณะเด่น (feature maps) P3, P4 และ P5 ที่ช่วยในการตรวจจับวัตถุในระดับขนาดต่าง ๆ

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
scales = ["P3 (stride 8): 80x80", "P4 (stride 16): 40x40", "P5 (stride 32): 20x20"]
grid_sizes = [80, 40, 20]

for idx, ax in enumerate(axes):
    grid_size = grid_sizes[idx]
    # Display dummy grid heatmap
    dummy_feat = np.random.uniform(0, 1, size=(grid_size, grid_size))
    im = ax.imshow(dummy_feat, cmap='inferno')
    ax.set_title(scales[idx], fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 4. ทำไม Decoupled Heads จึงมีความสำคัญ

ใน YOLO เวอร์ชันเก่า (จนถึง YOLOv5) หัวทำนาย (prediction head) เดี่ยวจะสร้างความน่าจะเป็นของคลาสและขอบเขตของกล่องพร้อมกันจากลักษณะเด่นที่ใช้งานร่วมกัน (shared features)

อย่างไรก็ตาม งานการจำแนกประเภท (classification) และการถดถอย (regression) เป็นงานที่ขัดแย้งกันในทางทฤษฎี:
-   **การจำแนกประเภท (Classification)** จะได้ประโยชน์จากการเรียนรู้ลักษณะเด่นเชิงพื้นที่ที่ไม่แปรเปลี่ยนตามการเลื่อนตำแหน่ง (translation-invariant spatial features) กล่าวคือจดจำพื้นผิวและรูปทรงได้ไม่ว่าจะอยู่ตรงไหนในเซลล์
-   **การถดถอย (Regression)** ต้องการการระบุตำแหน่งขอบเขตที่แปรเปลี่ยนตามการเลื่อนตำแหน่ง (translation-sensitive boundary localization) นั่นคือระบุขอบและตำแหน่งที่สอดคล้องกันอย่างแม่นยำ

การแยกหัวทำนายออกเป็นสองกิ่งคอนโวลูชันที่แตกต่างกัน (decoupled head) จะช่วยแก้ปัญหานี้ได้ ซึ่งช่วยปรับปรุงความเร็วในการลู่เข้า (convergence speed) ของการฝึกฝนแบบจำลอง และความแม่นยำสุดท้ายโดยรวม (mAP)